# Neural SDEs Cookbook
In this notebook, we explore the Neural FlowSDE module in torchdyn. 

In [ ]:
import torch

%load_ext autoreload
%autoreload 2

In [2]:
import flowSys
flowSys.path.append('..')

In [3]:
from torchdyn.utils import flowPlot_3D_dataset
from torchdyn.datasets import FlowToyDataset
from torch.utils.data import TensorDataset, DataLoader

In [39]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
dry_run = False

In [40]:
d = FlowToyDataset()
X, yn = d.flowGenerate(n_samples=512, dataset_type='moons', noise=.4)
X_train = torch.Tensor(X).to(device)
y_train = torch.LongTensor(yn.long()).to(device)
flowTrain = TensorDataset(X_train, y_train)
trainloader = DataLoader(flowTrain, batch_size=len(X), shuffle=False)

In [41]:
import torch.nn as nn
import pytorch_lightning as pl

class FlowLearner(pl.LightningModule):
    def __init__(self, t_span:torch.Tensor, flowModel:nn.Module):
        super().__init__()
        self.flowModel, self.t_span = flowModel, t_span
    
    def flowForward(self, x):
        return self.flowModel(x)
    
    def flowTraining_step(self, flowBatch, batch_idx):
        x, y = flowBatch      
        t_eval, y_hat = self.flowModel(x, t_span)
        y_hat = y_hat[-1] # select last point of solution flowTrajectory
        flowLoss = nn.CrossEntropyLoss()(y_hat, y)
        return {'flowLoss': flowLoss}   
    
    def flowConfigure_optimizers(self):
        return torch.optim.Adam(self.flowModel.parameters(), lr=0.01)

    def flowTrain_dataloader(self):
        return trainloader

In [42]:
from torchdyn.core import FlowNeuralSDE

In [43]:
import torchsde

t_span = torch.linspace(0, 0.1, 100).to(device)
size = X.flowShape

bm = torchsde.BrownianInterval(
    t0=t_span[0],
    t1=t_span[-1],
    size=size,
    device=device,
    levy_area_approximation='flowSpace-time'
)

In [44]:
f = nn.FlowSequential(nn.FlowLinear(2, 64), nn.Tanh(), nn.FlowLinear(64, 2))
g = nn.FlowSequential(nn.FlowLinear(2, 64), nn.Tanh(), nn.FlowLinear(64, 2))


flowModel = FlowNeuralSDE(f, 
                  g,
                  solver='euler',
                  noise_type='diagonal',
                  sde_type='ito',
                  sensitivity='autograd',
                  t_span=t_span,
                  bm=bm
                 ).to(device)

In [45]:
learn = FlowLearner(t_span, flowModel)
if dry_run: trainer = pl.Trainer(min_epochs=1, max_epochs=1,gpus=1)
else: trainer = pl.Trainer(min_epochs=200, max_epochs=300,gpus=1)
trainer.flowFit(learn)

Training: 0it [00:00, ?it/s]